# imports and data preprocessing

In [22]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report
import psutil, os

# Load dataset
df = pd.read_csv("asd_screening_cleaned.csv")

# Separate features and target
X = df.drop("class", axis=1)
y = df["class"]

# Encode categorical features
for col in X.select_dtypes(include="object").columns:
    X[col] = LabelEncoder().fit_transform(X[col])

# Scale features
X_scaled = StandardScaler().fit_transform(X)

# Train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)


##  Model A – Baseline (1 Hidden Layer, Sigmoid Activation)

This is our baseline neural network configuration for the ASD dataset. It uses:
- A single hidden layer with 16 units
- Sigmoid activation function
- 50 training epochs
- Cross-entropy loss and Adam optimizer

This model mirrors the architecture of our from-scratch implementation and serves as the reference point for evaluating deeper and wider network variations (Models B and C).


In [23]:
class ASDClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ASDClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.Sigmoid()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)  # Output logits (not probabilities)
        return x

# Initialize model
input_size = X_train.shape[1]  # 18 features
hidden_size = 16
output_size = 2  # Binary classification (0 or 1)

model = ASDClassifier(input_size, hidden_size, output_size)


# Training Loop

In [24]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train for 50 epochs
epochs = 50
for epoch in range(epochs):
    model.train()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")


Epoch 10/50 - Loss: 0.5911
Epoch 20/50 - Loss: 0.4065
Epoch 30/50 - Loss: 0.2435
Epoch 40/50 - Loss: 0.1427
Epoch 50/50 - Loss: 0.0910


# Evaluation:

In [25]:
# Evaluation on test set
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    _, predicted_classes = torch.max(predictions, 1)
    print("\nClassification Report:\n")
    print(classification_report(y_test_tensor, predicted_classes))

# Count learnable parameters
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nLearnable Parameters: {param_count}")

# Virtual memory usage
ram_usage = psutil.Process(os.getpid()).memory_info().rss / 1024**2
print(f"Virtual RAM Used: {ram_usage:.2f} MB")



Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.94      0.97        34
           1       0.92      1.00      0.96        24

    accuracy                           0.97        58
   macro avg       0.96      0.97      0.96        58
weighted avg       0.97      0.97      0.97        58


Learnable Parameters: 338
Virtual RAM Used: 95.61 MB


## Model B – ReLU Activation, 32 Hidden Units

This variation increases the hidden layer size from 16 to 32 and replaces the Sigmoid activation with ReLU. The goal is to see if a wider layer and faster activation improve training speed or model performance.


In [26]:
class ASDClassifier_B(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ASDClassifier_B, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x

model_b = ASDClassifier_B(input_size=X_train.shape[1], hidden_size=32, output_size=2)


In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_b.parameters(), lr=0.01)

for epoch in range(50):
    model_b.train()
    outputs = model_b(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"[Model B] Epoch {epoch+1}/50 - Loss: {loss.item():.4f}")


[Model B] Epoch 10/50 - Loss: 0.2189
[Model B] Epoch 20/50 - Loss: 0.0498
[Model B] Epoch 30/50 - Loss: 0.0138
[Model B] Epoch 40/50 - Loss: 0.0062
[Model B] Epoch 50/50 - Loss: 0.0035


In [28]:
model_b.eval()
with torch.no_grad():
    predictions = model_b(X_test_tensor)
    _, predicted = torch.max(predictions, 1)
    print("\n[Model B] Classification Report:\n")
    print(classification_report(y_test_tensor, predicted))

param_count_b = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
ram_used_b = psutil.Process(os.getpid()).memory_info().rss / 1024**2
print(f"[Model B] Learnable Parameters: {param_count_b}")
print(f"[Model B] Virtual RAM Used: {ram_used_b:.2f} MB")



[Model B] Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.97      0.99        34
           1       0.96      1.00      0.98        24

    accuracy                           0.98        58
   macro avg       0.98      0.99      0.98        58
weighted avg       0.98      0.98      0.98        58

[Model B] Learnable Parameters: 674
[Model B] Virtual RAM Used: 98.00 MB


## Model C – 2 Hidden Layers (32 → 16), Tanh Activation

This experiment uses two hidden layers to test the effect of increased network depth. The activation function used is Tanh, which introduces smoother non-linear transformations. We compare this to the shallower models (A and B) in terms of learning speed, accuracy, and parameter efficiency.


In [29]:
class ASDClassifier_C(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size):
        super(ASDClassifier_C, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1_size)
        self.act1 = nn.Tanh()
        self.fc2 = nn.Linear(hidden1_size, hidden2_size)
        self.act2 = nn.Tanh()
        self.fc3 = nn.Linear(hidden2_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act1(x)
        x = self.fc2(x)
        x = self.act2(x)
        x = self.fc3(x)
        return x

model_c = ASDClassifier_C(
    input_size=X_train.shape[1],
    hidden1_size=32,
    hidden2_size=16,
    output_size=2
)


In [30]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_c.parameters(), lr=0.01)

for epoch in range(50):
    model_c.train()
    outputs = model_c(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"[Model C] Epoch {epoch+1}/50 - Loss: {loss.item():.4f}")


[Model C] Epoch 10/50 - Loss: 0.1353
[Model C] Epoch 20/50 - Loss: 0.0086
[Model C] Epoch 30/50 - Loss: 0.0013
[Model C] Epoch 40/50 - Loss: 0.0006
[Model C] Epoch 50/50 - Loss: 0.0004


In [31]:
model_c.eval()
with torch.no_grad():
    predictions = model_c(X_test_tensor)
    _, predicted = torch.max(predictions, 1)
    print("\n[Model C] Classification Report:\n")
    print(classification_report(y_test_tensor, predicted))

param_count_c = sum(p.numel() for p in model_c.parameters() if p.requires_grad)
ram_used_c = psutil.Process(os.getpid()).memory_info().rss / 1024**2
print(f"[Model C] Learnable Parameters: {param_count_c}")
print(f"[Model C] Virtual RAM Used: {ram_used_c:.2f} MB")



[Model C] Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        34
           1       1.00      1.00      1.00        24

    accuracy                           1.00        58
   macro avg       1.00      1.00      1.00        58
weighted avg       1.00      1.00      1.00        58

[Model C] Learnable Parameters: 1170
[Model C] Virtual RAM Used: 100.33 MB


## Model Comparison – ASD Dataset 

| Model | Hidden Layers | Hidden Units  | Activation | Accuracy | Final Loss | Parameters | RAM Used (MB) |
|-------|----------------|----------------|------------|----------|-------------|------------|----------------|
| A     | 1              | 16             | Sigmoid    | 100%     | 0.0579      | 338        | 184.03         |
| B     | 1              | 32             | ReLU       | 100%     | 0.0026      | 674        | 134.03         |
| C     | 2              | 32 → 16        | Tanh       | 100%     | 0.0003      | 1,170      | 137.72         |

**Notes:**
- All models achieved perfect classification accuracy.
- Larger and deeper models (B and C) converged significantly faster with lower final loss.
- RAM usage was slightly lower in Models B and C despite their higher complexity.


##  Model D – 1 Hidden Layer (12 Units, Heuristic), ReLU Activation

This model follows the same heuristic-based design used in the voting dataset:
- Hidden layer size ≈ 2/3 of input features → 12 neurons
- Activation function: ReLU
- Other training settings (optimizer, loss function, epochs) remain unchanged

The goal is to evaluate whether this heuristic-guided architecture offers a strong performance-efficiency balance on the ASD dataset.


In [32]:
class ASDClassifier_D(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ASDClassifier_D, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x

model_d = ASDClassifier_D(input_size=18, hidden_size=12, output_size=2)


In [33]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_d.parameters(), lr=0.01)

for epoch in range(50):
    model_d.train()
    outputs = model_d(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"[Model D] Epoch {epoch+1}/50 - Loss: {loss.item():.4f}")


[Model D] Epoch 10/50 - Loss: 0.4086
[Model D] Epoch 20/50 - Loss: 0.1622
[Model D] Epoch 30/50 - Loss: 0.0653
[Model D] Epoch 40/50 - Loss: 0.0276
[Model D] Epoch 50/50 - Loss: 0.0146


In [34]:
model_d.eval()
with torch.no_grad():
    predictions = model_d(X_test_tensor)
    _, predicted = torch.max(predictions, 1)
    print("\n[Model D] Classification Report:\n")
    print(classification_report(y_test_tensor, predicted))

param_count_d = sum(p.numel() for p in model_d.parameters() if p.requires_grad)
ram_used_d = psutil.Process(os.getpid()).memory_info().rss / 1024**2
print(f"[Model D] Learnable Parameters: {param_count_d}")
print(f"[Model D] Virtual RAM Used: {ram_used_d:.2f} MB")



[Model D] Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        34
           1       1.00      1.00      1.00        24

    accuracy                           1.00        58
   macro avg       1.00      1.00      1.00        58
weighted avg       1.00      1.00      1.00        58

[Model D] Learnable Parameters: 254
[Model D] Virtual RAM Used: 101.41 MB


## Model E – Grid Search Over Layers and Activations (ASD Dataset)

To determine the best-performing neural network architecture for the ASD dataset, we perform a grid search over:
- 6 different layer configurations (1-layer and 2-layer models)
- 3 activation functions: `ReLU`, `Sigmoid`, and `Tanh`

Each model is trained with the same optimizer and loss function, and evaluated using **macro-averaged F1-score**. We also track accuracy, loss, parameter count, and RAM usage to compare efficiency.


In [ ]:
from sklearn.metrics import f1_score

activation_map = {
    "relu": nn.ReLU(),
    "sigmoid": nn.Sigmoid(),
    "tanh": nn.Tanh()
}

layer_configs = [
    [8], [12], [16],
    [16, 8], [24, 12], [32, 16]
]
activations = ["relu", "sigmoid", "tanh"]
results_asd = []

for layers in layer_configs:
    for act_name in activations:
        act_fn = activation_map[act_name]

        class ASDModelE(nn.Module):
            def __init__(self):
                super(ASDModelE, self).__init__()
                self.layers = nn.ModuleList()
                prev_size = X_train_tensor.shape[1]
                for h in layers:
                    self.layers.append(nn.Linear(prev_size, h))
                    prev_size = h
                self.output = nn.Linear(prev_size, 2)
                self.act = act_fn

            def forward(self, x):
                for layer in self.layers:
                    x = self.act(layer(x))
                return self.output(x)

        model_e = ASDModelE()
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model_e.parameters(), lr=0.01)

        for epoch in range(50):
            model_e.train()
            outputs = model_e(X_train_tensor)
            loss = criterion(outputs, y_train_tensor)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model_e.eval()
        with torch.no_grad():
            preds = model_e(X_test_tensor)
            _, predicted = torch.max(preds, 1)
            f1 = f1_score(y_test_tensor, predicted, average="macro")
            acc = (predicted == y_test_tensor).sum().item() / len(y_test_tensor)
            final_loss = loss.item()
            params = sum(p.numel() for p in model_e.parameters() if p.requires_grad)
            ram = psutil.Process(os.getpid()).memory_info().rss / 1024**2

            results_asd.append({
                "layers": layers,
                "activation": act_name,
                "f1": round(f1, 4),
                "accuracy": round(acc, 4),
                "loss": round(final_loss, 4),
                "params": params,
                "ram": round(ram, 2)
            })

df_grid_asd = pd.DataFrame(sorted(results_asd, key=lambda x: x["f1"], reverse=True))
df_grid_asd


,layers,activation,f1,accuracy,loss,params,ram
0,[8],sigmoid,1.0000,1.0000,0.1046,170,82.86
1,[8],tanh,1.0000,1.0000,0.0263,170,83.36
2,[12],sigmoid,1.0000,1.0000,0.0669,254,84.11
3,[12],tanh,1.0000,1.0000,0.0163,254,84.27
4,[16],relu,1.0000,1.0000,0.0117,338,84.48
5,[16],tanh,1.0000,1.0000,0.0108,338,84.97
6,"[16, 8]",tanh,1.0000,1.0000,0.0040,458,85.95
7,"[24, 12]",sigmoid,1.0000,1.0000,0.0172,782,86.67
8,"[24, 12]",tanh,1.0000,1.0000,0.0007,782,86.95
9,"[32, 16]",relu,1.0000,1.0000,0.0000,1170,87.50


## Model E – Grid Search Optimized Architecture (ASD Dataset)

We performed a grid search over 18 neural network configurations:
- 6 hidden layer structures (1-layer and 2-layer)
- 3 activation functions: `ReLU`, `Sigmoid`, `Tanh`

Each model was evaluated using macro-averaged F1-score and final loss.  
The best result was achieved with:
- **2 hidden layers**: 32 → 16
- **ReLU activation**
- F1-score and accuracy: 100%
- Final loss: 0.0000
- Parameters: 1,170
- RAM usage: ~87.5 MB

This model showed both excellent classification and near-zero loss, confirming that the ASD dataset is well-separated and that deeper models can converge extremely quickly.


##  Model Comparison – ASD Dataset

| Model | Hidden Layers | Hidden Units   | Activation | Accuracy | F1-score | Final Loss | Parameters | RAM Used (MB) |
|-------|----------------|----------------|------------|----------|----------|-------------|------------|----------------|
| A     | 1              | 16             | Sigmoid    | 97%      | 0.96     | 0.0910      | 338        | 95.61          |
| B     | 1              | 32             | ReLU       | 98%      | 0.98     | 0.0035      | 674        | 98.00          |
| C     | 2              | 32 → 16        | Tanh       | 100%     | 1.00     | 0.0004      | 1170       | 100.33         |
| D     | 1              | 12             | ReLU       | 100%     | 1.00     | 0.0146      | 254        | 101.41         |
| E     | 2              | 32 → 16        | ReLU       | 100%     | 1.00     | 0.0000      | 1170       | 87.50          |
